# 01 — Audit Data

Memeriksa apa yang benar-benar ada di direktori `data/` sebelum satu baris
model pun ditulis.

Ringkasan lengkapnya ada di `docs/DATA_AUDIT.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from backend.app.core.config import get_settings
settings = get_settings()
print("Akar proyek:", settings.paths.root)
print("Mode deployment:", settings.deployment_mode)

## Isi direktori data

In [ ]:
for path in sorted(settings.paths.raw_data.iterdir()):
    print(f"{path.stat().st_size / 1_048_576:8.2f} MB  {path.name}")

## Bentuk berkas sumber

Nama kedua sheet mengandung spasi di ujung. Itu apa adanya di berkas
sumber; merapikannya membuat pembacaan gagal.

In [ ]:
import openpyxl
from backend.app.data.event_etl import PRIMARY_SOURCE_GLOB, find_source_file

source = find_source_file(settings.paths.raw_data, PRIMARY_SOURCE_GLOB)
workbook = openpyxl.load_workbook(source, read_only=True)
print(source.name)
print(f"{len(workbook.sheetnames)} sheet")
for name in workbook.sheetnames:
    if "UNIT 1" in name.upper():
        print(f"  {name!r}")
workbook.close()

## Sepuluh baris pertama sheet derating

In [ ]:
from backend.app.core.constants import SHEET_DERATING

pd.read_excel(source, sheet_name=SHEET_DERATING, header=None, nrows=8).iloc[:, :11]

## Membangun Event Registry

In [ ]:
from backend.app.data.event_etl import build_event_registry

registry, report = build_event_registry(settings)
print(report.render())

In [ ]:
print("Total event:", len(registry))
print(registry["record_kind"].value_counts().to_string())
print()
print("Rentang:", registry["start_time"].min(), "s/d", registry["start_time"].max())

## Kolom yang tidak ada di sumber

Skema README §15 meminta `coal_source`, `coal_blending`, dan
`clinker_found`. Ketiganya tidak ada di berkas sumber dan dibiarkan
kosong — menebaknya akan merusak analisis pengaruh kualitas batubara.

In [ ]:
missing = ["coal_source", "coal_blending", "clinker_found", "operator_action"]
registry[missing].notna().sum().to_frame("terisi")

## Uji konsistensi antar-snapshot April dan Mei 2026

In [ ]:
from backend.app.data.event_etl import cross_check_sources

result = cross_check_sources(settings)
for key in ("old_file", "new_file", "old_rows", "new_rows", "rows_added",
            "rows_missing_from_new", "consistent"):
    print(f"{key:24}: {result[key]}")

## Tag DCS

Seluruh entri masih kosong. Inilah kendala utama Fase 0.

In [ ]:
print("Tag Priority A terdaftar :", len(settings.standard_names("priority_a")))
print("Tag Priority B terdaftar :", len(settings.standard_names("priority_b")))
print("Tag DCS sudah dipetakan  :", len(settings.dcs_tag_lookup()))